In [ ]:
pip install requests beautifulsoup4 pandas


In [16]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

base_url = 'https://lifehacker.ru/topics/technology/'  # базовая часть ссылки

# Создадим список для хранения ссылок на статьи
links = []

# Получаем содержимое 10 страниц списка материалов
for page_num in range(1, 11):
    url = f'{base_url}?page={page_num}'  # используем пагинацию
    response = requests.get(url)

    if response.status_code != 200:
        print(f'Ошибка при получении страницы {page_num}: {response.status_code}')
        continue  # переходим к следующей итерации, если страница не доступна

    soup = BeautifulSoup(response.text, 'lxml')

    # Находим все карточки статей
    raw_items = soup.find_all('a', class_='lh-small-article-card__link')

    # Извлекаем ссылки
    for item in raw_items:
        link = item.get('href')
        if link:
            # Полная ссылка на статью
            full_link = f'https://lifehacker.ru{link}' if link.startswith('/') else link
            links.append(full_link)

# Теперь переберем все полученные ссылки и получим html-код каждого материала
result = []
for url in tqdm(links):
    article = {}
    response = requests.get(url)

    if response.status_code != 200:
        print(f'Ошибка при получении материала: {url} - {response.status_code}')
        continue  # переходим к следующему материалу, если он не доступен

    soup = BeautifulSoup(response.text, 'lxml')

    # Извлекаем заголовок с использованием атрибутов name и aria-label
    title = soup.find('h1', class_='article-card__title')  # Заголовок статьи, попробуем просто найти h1
    if title:
        article['title'] = title.text.strip()  # Заголовок статьи
    else:
        print(f'Заголовок не найден для: {url}')
        continue

    # Извлекаем текст содержимого статьи
    content = soup.find('div', class_='article-card__subtitle')  # Попробуем найти основной текст
    if content:
        article['text'] = content.text.strip()  # Содержимое статьи
    else:
        print(f'Содержимое не найдено для: {url}')
        continue

    result.append(article)




 69%|██████▉   | 208/300 [03:04<00:55,  1.65it/s]

Заголовок не найден для: https://lifehacker.ru/special/msi-your-laptop/


100%|██████████| 300/300 [04:23<00:00,  1.14it/s]


In [19]:
# Создание DataFrame только с заголовками и текстом
df = pd.DataFrame(result, columns=['title', 'text'])

# Вывод таблицы с заголовками и текстом
print(df)


                                                                                     title  \
0                  Tecno представила карманную консоль Pocket Go с AR-очками вместо экрана   
1                                                                    Как выбрать ирригатор   
2             В России создали «распределяющую шляпу». Она поможет выбрать колледж или вуз   
3                            Реальную автономность всех версий iPhone 16 сравнили на видео   
4    Обзор Dreame Z30 — мощного вертикального пылесоса с насадкой для вычёсывания питомцев   
..                                                                                     ...   
294                       Как быстро сделать фото на документы с нейросетью Passport Maker   
295                                  15 крутых бюджетных смартфонов, которые не разочаруют   
296                        10 бесплатных нейросетей для генерации текстов на русском языке   
297                                            3 гаджета от 